In [10]:
import mlflow
import dagshub
import joblib

In [11]:
# Dags Hub Variáveis
REPO_OWNER = "JosueJNLui"
REPO_NAME = "fiap-mlet-challenge-fase-1"

# Nome do experimento no MLflow
EXPERIMENT_NAME = "Churn-Predict-Telco"

In [12]:
# Configuração de Rastreabilidade atrabés do MLflow + DagsHub
dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME, mlflow=True)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

Initialized MLflow to track repo "JosueJNLui/fiap-mlet-challenge-fase-1"

Repository JosueJNLui/fiap-mlet-challenge-fase-1 initialized!

## Comparando as métricas frente ao Dataset de Validação - Treino vs Validação

In [13]:
modelos = ["DummyClassifier_kfold_validation", "LogisticRegression_kfold_validation", "RandomForest_kfold_validation", 
            "MLP_lr=0.01_dropout=0.15_batch=128_hidden_dims=8", # MLP_8_lr0.01_drp0.15_b128 
            "MLP_lr=0.001_dropout=0.3_batch=32_hidden_dims=32", # MLP_32_lr0.001_drp0.3_b32
            "MLP_lr=0.01_dropout=0.3_batch=64_hidden_dims=8"    # MLP_8_lr0.01_drp0.3_b64
            ]

In [14]:
# Buscamos apenas apenas as Runs desejadas (top 3 MLPs + baselines)
df_runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

df_final = df_runs.set_index('tags.mlflow.runName').loc[modelos]

# Selecionando apenas o essencial para a monografia
tabela_tese = df_final[[
    'metrics.roc_auc', 
    'metrics.pr_auc',
    'metrics.recall',
    'metrics.f1_score',
    'metrics.accuracy',
    'metrics.precision',
    'metrics.lucro_liquido_BRL', 
    'metrics.custo_churn_perdido_BRL',
    'metrics.custo_falso_positivo_BRL'
]]

display(tabela_tese)

,metrics.roc_auc,metrics.pr_auc,metrics.recall,metrics.f1_score,metrics.accuracy,metrics.precision,metrics.lucro_liquido_BRL,metrics.custo_churn_perdido_BRL,metrics.custo_falso_positivo_BRL
tags.mlflow.runName,,,,,,,,,
DummyClassifier_kfold_validation,0.500000,0.632676,0.000000,0.000000,0.734647,0.000000,-74750.0,74750.0,0.0
LogisticRegression_kfold_validation,0.849513,0.670088,0.964523,0.565084,0.601729,0.401101,33120.0,2650.0,21910.0
RandomForest_kfold_validation,0.841290,0.656566,0.957199,0.563662,0.603483,0.400562,32340.0,3200.0,21700.0
MLP_lr=0.01_dropout=0.15_batch=128_hidden_dims=8,0.847806,0.668916,0.956515,0.573265,0.618751,0.410650,33120.0,3250.0,20830.0
MLP_lr=0.001_dropout=0.3_batch=32_hidden_dims=32,0.846366,0.669048,0.959893,0.568297,0.611282,0.404296,33100.0,3000.0,21300.0
MLP_lr=0.01_dropout=0.3_batch=64_hidden_dims=8,0.849074,0.672938,0.955817,0.572711,0.617693,0.410464,32980.0,3300.0,20880.0


### Embora as arquiteturas de redes neurais (MLP) tenham apresentado métricas de precisão e F1-Score levemente superiores, a Regressão Logística foi selecionada como o modelo de produção final. Esta decisão fundamenta-se no princípio da parcimônia: diante de um empate técnico no lucro líquido médio (R$ 33.120,00) validado pelo teste de Nemenyi, a Regressão Logística oferece maior interpretabilidade dos fatores determinantes do churn e menor custo operacional de manutenção e deploy (MLOps), garantindo uma solução robusta e transparente para as estratégias de retenção da companhia

## Comparando as métricas frente ao Dataset de Teste - (Treino + Validação) vs Teste

In [16]:
modelos = ["DummyClassifier", "LogisticRegression", "RandomForest", "MLP_GridSearch_KFold"]

In [17]:
# Buscamos apenas apenas as Runs desejadas (top 3 MLPs + baselines)
df_runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

df_final = df_runs.set_index('tags.mlflow.runName').loc[modelos]

# Selecionando apenas o essencial para a monografia
tabela_tese = df_final[[
    'metrics.roc_auc', 
    'metrics.recall',
    'metrics.f1_score',
    'metrics.accuracy',
    'metrics.precision',
    'metrics.lucro_liquido_BRL', 
    'metrics.custo_churn_perdido_BRL',
    'metrics.custo_falso_positivo_BRL',
]]

display(tabela_tese)

,metrics.roc_auc,metrics.recall,metrics.f1_score,metrics.accuracy,metrics.precision,metrics.lucro_liquido_BRL,metrics.custo_churn_perdido_BRL,metrics.custo_falso_positivo_BRL
tags.mlflow.runName,,,,,,,,
DummyClassifier,0.500000,0.000000,0.000000,0.734564,0.000000,-187000.0,187000.0,0.0
LogisticRegression,0.849089,0.959893,0.560062,0.599716,0.395374,81200.0,7500.0,54900.0
RandomForest,0.840492,0.954545,0.550926,0.586941,0.387202,77800.0,8500.0,56500.0
MLP_GridSearch_KFold,0.846648,0.973262,0.540460,0.560681,0.374101,79700.0,5000.0,60900.0


### A avaliação final no conjunto de teste (hold-out) ratificou a Regressão Logística como o modelo superior. Enquanto a Rede Neural (MLP) demonstrou uma tendência ao aumento de falsos positivos quando exposta a dados inéditos — elevando os custos operacionais de retenção para R$ 60.900,00 — a Regressão Logística manteve sua estabilidade estatística. O modelo linear alcançou o maior lucro líquido final (R$ 81.200,00), superando a MLP em 1,8% e a Random Forest em 4,3%, provando que, para este problema de negócio, a simplicidade estrutural resultou em uma capacidade de generalização financeira superior.